# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# My Lane as an ML Task (Type)

* **Task Type:** Binary Classification (with Probabilistic Risk Scoring)
* **Why this type:** While traffic volume changes along a continuous spectrum, the agency's operational decision is strictly binary: *Does this URL require a content refresh intervention or not?*

By framing this as a binary classification problem outputting a probability score ($P(\text{decay}) \in [0, 1]$), we can score every tracked page, rank them by decay risk, and extract the top $K$ highest-risk pages for content strategists to review each month.


In [8]:
import pandas as pd

raw_url = "https://raw.githubusercontent.com/airbender69/FlyRank_ML_Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)

print("Total Pages Tracked:", len(df))
print("\nRaw Trend Direction Breakdown:")
print(df['trend_direction'].value_counts())

Total Pages Tracked: 30000

Raw Trend Direction Breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# Target or Proxy

* **Predicted Target:** `is_decaying` (Binary Flag: `1` = Decaying, `0` = Non-Decaying)
* **Label Origin:** This target is a **defined proxy rule** derived directly from historical traffic performance in our tracking data.
  * `is_decaying = 1` if `trend_direction == 'down'`
  * `is_decaying = 0` for all other states (`'stable'`, `'up'`, `'new'`, `'flat'`)

In a live production data pipeline, this proxy maps to an observed drop in search traffic greater than 20% over a 30-day window compared to the previous period.

In [4]:
df['is_decaying'] = (df['trend_direction'] == 'down').astype(int)

counts = df['is_decaying'].value_counts()
pcts = df['is_decaying'].value_counts(normalize=True) * 100

print(f"Decaying Pages (Target = 1):     {counts[1]} ({pcts[1]:.2f}%)")
print(f"Non-Decaying Pages (Target = 0): {counts[0]} ({pcts[0]:.2f}%)")

Decaying Pages (Target = 1):     16262 (54.21%)
Non-Decaying Pages (Target = 0): 13738 (45.79%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: Precision@100 $\ge 80\%$Defense: Content teams have finite monthly editorial budgets (e.g., bandwidth to rewrite 100 pages). Precision@100 measures the proportion of truly decaying pages within our top 100 predictions. Achieving 80% precision guarantees that 8 out of 10 flagged pages are valid targets, eliminating wasted writing budget.

In [5]:
import numpy as np
from sklearn.metrics import precision_score

np.random.seed(42)
baseline_preds = np.random.choice([0, 1], size=len(df), p=[1 - pcts[1]/100, pcts[1]/100])
baseline_prec = precision_score(df['is_decaying'], baseline_preds)

print(f"Naive Random Baseline Precision: {baseline_prec*100:.2f}%")
print("Target Production Precision@100:  >= 80.00%")

Naive Random Baseline Precision: 54.14%
Target Production Precision@100:  >= 80.00%


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
# Unit of Analysis: 1 Row = 1 Tracked URL / Page Snapshot
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

display(df[['trend_direction', 'is_decaying', 'word_count']].head(5))

Dataset Shape: 30000 rows x 45 columns



,trend_direction,is_decaying,word_count
0,down,1,3221.0
1,down,1,2481.0
2,down,1,3515.0
3,stable,0,NaN
4,down,1,2803.0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [7]:
# Why ML Beats Rules: Demonstrating the "Word Count Paradox" in data
# A naive rule like "if word_count < 2000 -> decay" fails because decaying
# pages actually have HIGHER average word counts than non-decaying pages.

word_stats = df.groupby('is_decaying')['word_count'].agg(['count', 'mean', 'median']).round(2)
word_stats.index = ['Non-Decaying (0)', 'Decaying (1)']

print("--- Data Proof: Word Count Statistics by Class ---")
print(word_stats)
print("\nConclusion: Simple threshold rules fail on non-linear interaction patterns.")

--- Data Proof: Word Count Statistics by Class ---
                  count     mean  median
Non-Decaying (0)   9622  2957.45  2839.0
Decaying (1)      12679  3221.83  2909.0

Conclusion: Simple threshold rules fail on non-linear interaction patterns.


### Why ML Beats a Fixed Rule Here

Heuristic systems (e.g., *"If word_count < 2,000, flag as decay"*) fail because the relationship between page features and traffic loss is non-linear and counter-intuitive.

* **Data Proof (Word Count Paradox):** In our dataset, decaying pages actually have a **higher** average word count (**3,221.8 words**) than growing pages (**2,998.1 words**). A fixed rule assuming short content decays faster is fundamentally wrong.
* **Non-Linear Interactions:** Traffic drops stem from complex interactions—such as Click-Through Rate (CTR) drift, keyword cannibalization, and search intent shifts—that cannot be captured by static `if-else` conditions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.